# **BLOCK 1: SETUP, UNZIP DATASET, AND VERIFICATION**

In [3]:
# ============================================================================
# BLOCK 1: SETUP PATHS FOR SHOULDER/ARM MODEL
# ============================================================================

import os
import json
import yaml
import glob
import time

print("="*70)
print("SHOULDER & ARM FRACTURE DETECTION - FASTER R-CNN")
print("="*70)

# ========== YOUR DATASET PATHS ==========
BASE_PATH = "/kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL"

# Paths to your splits
TRAIN_IMAGES = os.path.join(BASE_PATH, "train", "images")
TRAIN_LABELS = os.path.join(BASE_PATH, "train", "labels")
VAL_IMAGES = os.path.join(BASE_PATH, "val", "images")
VAL_LABELS = os.path.join(BASE_PATH, "val", "labels")
TEST_IMAGES = os.path.join(BASE_PATH, "test", "images")
TEST_LABELS = os.path.join(BASE_PATH, "test", "labels")

WORKING_DIR = "/kaggle/working"

print(f"📁 Base path: {BASE_PATH}")
print(f"📁 Train images: {TRAIN_IMAGES}")
print(f"📁 Train labels: {TRAIN_LABELS}")
print(f"📁 Val images: {VAL_IMAGES}")
print(f"📁 Test images: {TEST_IMAGES}")
print(f"📁 Working dir: {WORKING_DIR}")

# Verify paths exist
for path, name in [(TRAIN_IMAGES, "Train images"), (TRAIN_LABELS, "Train labels"),
                   (VAL_IMAGES, "Val images"), (VAL_LABELS, "Val labels"),
                   (TEST_IMAGES, "Test images"), (TEST_LABELS, "Test labels")]:
    exists = os.path.exists(path)
    print(f"  {name}: {'✅' if exists else '❌'} {path if exists else 'Not found'}")

# Count images
train_count = len([f for f in os.listdir(TRAIN_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))])
val_count = len([f for f in os.listdir(VAL_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))])
test_count = len([f for f in os.listdir(TEST_IMAGES) if f.endswith(('.jpg', '.png', '.jpeg'))])

print(f"\n📊 DATASET STATISTICS:")
print(f"   Training: {train_count} images")
print(f"   Validation: {val_count} images")
print(f"   Test: {test_count} images")
print(f"   TOTAL: {train_count + val_count + test_count} images")

print("="*70)

SHOULDER & ARM FRACTURE DETECTION - FASTER R-CNN
📁 Base path: /kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL
📁 Train images: /kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL/train/images
📁 Train labels: /kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL/train/labels
📁 Val images: /kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL/val/images
📁 Test images: /kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL/test/images
📁 Working dir: /kaggle/working
  Train images: ✅ /kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL/train/images
  Train labels: ✅ /kaggle/input/datasets/chatecu/arm-and-shoulder-fracture-dataset/Enhanced_Shoulder_Arm_Dataset_FINAL/train/labels
  Val images: ✅ /kaggle/inpu

# **BLOCK 2: INSTALL FASTER R-CNN DEPENDENCIES**

In [2]:
# ============================================================================
# BLOCK 2: INSTALL DEPENDENCIES
# ============================================================================

print("="*70)
print("INSTALLING DEPENDENCIES")
print("="*70)

!pip install -q pandas pyyaml scikit-learn tqdm matplotlib opencv-python
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2
from PIL import Image

print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("="*70)

INSTALLING DEPENDENCIES
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.9 MB/s eta 0:00:00

✅ PyTorch: 2.10.0+cu128
✅ CUDA Available: True
✅ GPU: Tesla T4
✅ GPU Memory: 15.6 GB


# **BLOCK 3: CONVERT YOLO TO COCO FORMAT (FOR FASTER R-CNN)**

In [4]:
# ============================================================================
# BLOCK 3: CONVERT YOLO TO COCO FORMAT (WITH CORRECT CLASS IDS)
# ============================================================================

print("="*70)
print("CONVERTING YOLO TO COCO FORMAT")
print("="*70)

import json
from PIL import Image
from tqdm import tqdm

def yolo_to_coco(images_dir, labels_dir, output_json):
    """
    Convert YOLO format to COCO JSON format
    All classes become class 0 (fracture)
    """
    if not os.path.exists(images_dir):
        print(f"⚠️ Images directory not found: {images_dir}")
        return None
    
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [{"id": 1, "name": "fracture"}]
    }
    
    annotation_id = 1
    image_id = 1
    
    image_files = [f for f in os.listdir(images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"🔄 Converting {len(image_files)} images...")
    
    for img_file in tqdm(image_files, desc="Converting"):
        img_path = os.path.join(images_dir, img_file)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_path = os.path.join(labels_dir, label_file)
        
        if not os.path.exists(label_path):
            continue
        
        # Get image dimensions
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except Exception as e:
            tqdm.write(f"⚠️ Could not read {img_file}: {e}")
            continue
        
        # Add image
        coco_data["images"].append({
            "id": image_id,
            "file_name": img_file,
            "width": width,
            "height": height
        })
        
        # Add annotations (all converted to class 0)
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    # Read YOLO format: class x_center y_center width height
                    # We ignore the original class and use 0
                    x_center = float(parts[1]) * width
                    y_center = float(parts[2]) * height
                    bbox_width = float(parts[3]) * width
                    bbox_height = float(parts[4]) * height
                    
                    x = x_center - bbox_width/2
                    y = y_center - bbox_height/2
                    
                    coco_data["annotations"].append({
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": 0,  # ALWAYS class 0 (fracture)
                        "bbox": [x, y, bbox_width, bbox_height],
                        "area": bbox_width * bbox_height,
                        "iscrowd": 0
                    })
                    annotation_id += 1
        
        image_id += 1
    
    with open(output_json, 'w') as f:
        json.dump(coco_data, f, indent=2)
    
    print(f"✅ Converted {len(coco_data['images'])} images, {len(coco_data['annotations'])} annotations")
    print(f"   Categories: {coco_data['categories']}")
    return coco_data

# Convert each split
train_json = f"{WORKING_DIR}/train_coco.json"
val_json = f"{WORKING_DIR}/val_coco.json"
test_json = f"{WORKING_DIR}/test_coco.json"

print("\n📊 Converting TRAIN split...")
yolo_to_coco(TRAIN_IMAGES, TRAIN_LABELS, train_json)

print("\n📊 Converting VALIDATION split...")
yolo_to_coco(VAL_IMAGES, VAL_LABELS, val_json)

print("\n📊 Converting TEST split...")
yolo_to_coco(TEST_IMAGES, TEST_LABELS, test_json)

print("\n✅ COCO conversion complete!")
print("="*70)

CONVERTING YOLO TO COCO FORMAT

📊 Converting TRAIN split...
🔄 Converting 18987 images...


Converting: 100%|██████████| 18987/18987 [04:15<00:00, 74.39it/s]


✅ Converted 18987 images, 21304 annotations
   Categories: [{'id': 0, 'name': 'fracture'}]

📊 Converting VALIDATION split...
🔄 Converting 4832 images...


Converting: 100%|██████████| 4832/4832 [01:13<00:00, 65.96it/s]


✅ Converted 4832 images, 5245 annotations
   Categories: [{'id': 0, 'name': 'fracture'}]

📊 Converting TEST split...
🔄 Converting 3661 images...


Converting: 100%|██████████| 3661/3661 [00:58<00:00, 62.19it/s]

✅ Converted 3661 images, 4124 annotations
   Categories: [{'id': 0, 'name': 'fracture'}]

✅ COCO conversion complete!


# **BLOCK 4: REGISTER DATASET WITH DETECTRON2**

In [5]:
# ============================================================================
# BLOCK 4: REGISTER DATASET WITH DETECTRON2 (FIXED)
# ============================================================================

print("="*70)
print("REGISTERING DATASET WITH DETECTRON2")
print("="*70)

from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.utils.logger import setup_logger

setup_logger()

# Create unique dataset name with timestamp
timestamp = str(int(time.time()))[-8:]
dataset_name = f"shoulder_arm_{timestamp}"

print(f"📝 Dataset name: {dataset_name}")

# Register datasets
register_coco_instances(f"{dataset_name}_train", {}, train_json, TRAIN_IMAGES)
register_coco_instances(f"{dataset_name}_val", {}, val_json, VAL_IMAGES)
register_coco_instances(f"{dataset_name}_test", {}, test_json, TEST_IMAGES)

# Set metadata (only 1 class: fracture)
# IMPORTANT: category_id must be 0 to match the COCO file
class_names = ['fracture']
MetadataCatalog.get(f"{dataset_name}_train").thing_classes = class_names
MetadataCatalog.get(f"{dataset_name}_val").thing_classes = class_names
MetadataCatalog.get(f"{dataset_name}_test").thing_classes = class_names

# Also set category_id mapping (0 -> 0)
for name in [f"{dataset_name}_train", f"{dataset_name}_val", f"{dataset_name}_test"]:
    MetadataCatalog.get(name).thing_dataset_id_to_contiguous_id = {0: 0}

# Verify registration
print("\n📊 Registered datasets:")
for split in ['train', 'val', 'test']:
    name = f"{dataset_name}_{split}"
    if name in DatasetCatalog.list():
        dataset = DatasetCatalog.get(name)
        print(f"  ✅ {name}: {len(dataset)} images")
    else:
        print(f"  ❌ {name} not found!")

# Save dataset name for later
with open(f"{WORKING_DIR}/dataset_name.txt", 'w') as f:
    f.write(dataset_name)

print("\n✅ Dataset registration complete!")
print("="*70)

REGISTERING DATASET WITH DETECTRON2
📝 Dataset name: shoulder_arm_76159387

📊 Registered datasets:
WARNING [04/14 09:36:27 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[04/14 09:36:27 d2.data.datasets.coco]: Loaded 18987 images in COCO format from /kaggle/working/train_coco.json
  ✅ shoulder_arm_76159387_train: 18987 images
WARNING [04/14 09:36:27 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[04/14 09:36:27 d2.data.datasets.coco]: Loaded 4832 images in COCO format from /kaggle/working/val_coco.json
  ✅ shoulder_arm_76159387_val: 4832 images
WARNING [04/14 09:36:27 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[04/14 09:36:27 d2.data.datasets.coco]: Loaded 3661 images in COCO format from /kaggle/working/test_coco.json
  ✅ shoulder_arm_76159387_test: 3661 images

✅ Dataset registration 

# **BLOCK 5: CONFIGURE FASTER R-CNN (ENHANCED)**

In [6]:
# ============================================================================
# BLOCK 5: CONFIGURE FASTER R-CNN (ENHANCED FOR 21k IMAGES)
# ============================================================================

print("="*70)
print("CONFIGURING FASTER R-CNN MODEL (ENHANCED)")
print("="*70)

from detectron2.config import get_cfg
from detectron2 import model_zoo

# Load dataset name
with open(f"{WORKING_DIR}/dataset_name.txt", 'r') as f:
    dataset_name = f.read().strip()

print(f"📊 Using dataset: {dataset_name}")

# Create configuration
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))

# Dataset
cfg.DATASETS.TRAIN = (f"{dataset_name}_train",)
cfg.DATASETS.TEST = (f"{dataset_name}_val",)

# Classes (only fracture)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

# ========== ENHANCED TRAINING PARAMETERS ==========
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.000125
cfg.SOLVER.MAX_ITER = 37500  # CHANGED: 12 epochs (was 25000)
cfg.SOLVER.STEPS = (25000, 30000)  # CHANGED: LR decay at 67% and 80%
cfg.SOLVER.WEIGHT_DECAY = 0.0005
cfg.SOLVER.CHECKPOINT_PERIOD = 2500

# ========== ENHANCEMENTS ==========
# Warmup iterations
cfg.SOLVER.WARMUP_ITERS = 500
cfg.SOLVER.WARMUP_FACTOR = 0.001

# ========== DATA AUGMENTATION (NEW) ==========
cfg.INPUT.RANDOM_FLIP = "horizontal"
cfg.INPUT.RANDOM_ROTATION = 10  # Small rotations for X-rays
cfg.INPUT.BRIGHTNESS = 0.2  # Random brightness variation
cfg.INPUT.CONTRAST = 0.2  # Random contrast variation

# Add this to your training config to see AP50 every 2500 iterations
cfg.SOLVER.CHECKPOINT_PERIOD = 2500
# Then check the output logs for validation AP50 at each checkpoint

# ========== FASTER R-CNN SPECIFIC ==========
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.5

# Multi-scale anchors for different bone sizes
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[32, 64, 128, 256, 512]]

# Input size (standardized)
cfg.INPUT.MIN_SIZE_TRAIN = (800,)
cfg.INPUT.MAX_SIZE_TRAIN = 800
cfg.INPUT.MIN_SIZE_TEST = 800
cfg.INPUT.MAX_SIZE_TEST = 800

# Output directory
cfg.OUTPUT_DIR = f"{WORKING_DIR}/shoulder_arm_model"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print(f"\n⚙️ ENHANCED CONFIGURATION:")
print(f"  🏗️ Model: Faster R-CNN with ResNet-50 FPN")
print(f"  📋 Classes: {cfg.MODEL.ROI_HEADS.NUM_CLASSES}")
print(f"  📦 Batch size: {cfg.SOLVER.IMS_PER_BATCH}")
print(f"  🎯 Max iterations: {cfg.SOLVER.MAX_ITER} (12 epochs)")
print(f"  📉 Learning rate: {cfg.SOLVER.BASE_LR}")
print(f"  📉 LR decay at: {cfg.SOLVER.STEPS}")
print(f"  🔥 Warmup iterations: {cfg.SOLVER.WARMUP_ITERS}")
print(f"  💾 Checkpoint period: {cfg.SOLVER.CHECKPOINT_PERIOD}")
print(f"  🔄 Augmentations: RandomFlip, Rotation ±10°, Brightness/Contrast ±20%")
print(f"  📁 Output: {cfg.OUTPUT_DIR}")

# Save config
with open(f"{cfg.OUTPUT_DIR}/config.yaml", 'w') as f:
    f.write(cfg.dump())

print("\n✅ Configuration saved!")
print("="*70)

CONFIGURING FASTER R-CNN MODEL (ENHANCED)
📊 Using dataset: shoulder_arm_76159387

⚙️ ENHANCED CONFIGURATION:
  🏗️ Model: Faster R-CNN with ResNet-50 FPN
  📋 Classes: 1
  📦 Batch size: 4
  🎯 Max iterations: 37500 (12 epochs)
  📉 Learning rate: 0.000125
  📉 LR decay at: (25000, 30000)
  🔥 Warmup iterations: 500
  💾 Checkpoint period: 2500
  🔄 Augmentations: RandomFlip, Rotation ±10°, Brightness/Contrast ±20%
  📁 Output: /kaggle/working/shoulder_arm_model

✅ Configuration saved!


# **BLOCK 6: TRAIN FASTER R-CNN (ENHANCED WITH EARLY STOPPING)**

In [7]:
# ============================================================================
# BLOCK 6: TRAIN FASTER R-CNN MODEL (ENHANCED - FIXED)
# ============================================================================

print("="*70)
print("TRAINING FASTER R-CNN MODEL (ENHANCED)")
print("⚠️ This will take 5-6 hours!")
print("="*70)

import glob
import re
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
from detectron2.checkpoint import DetectionCheckpointer

class EnhancedTrainer(DefaultTrainer):
    """
    Enhanced trainer with early stopping and better evaluation
    """
    
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, cfg, True, output_folder)
    
    def __init__(self, cfg):
        super().__init__(cfg)
        self.best_val_ap = 0.0
        self.patience_counter = 0
        self.patience = 5  # Stop if no improvement for 5 evaluations
        self.dataset_name = None
        
        # Load dataset name
        with open(f"{WORKING_DIR}/dataset_name.txt", 'r') as f:
            self.dataset_name = f.read().strip()
    
    def after_step(self):
        """Called after each iteration"""
        super().after_step()
        
        # Check if we should evaluate (every CHECKPOINT_PERIOD)
        if self.iter % self.cfg.SOLVER.CHECKPOINT_PERIOD == 0 and self.iter > 0:
            self.evaluate_and_check_early_stop()
    
    def evaluate_and_check_early_stop(self):
        """Evaluate on validation set and check for early stopping"""
        from detectron2.evaluation import inference_on_dataset
        from detectron2.data import build_detection_test_loader
        
        print(f"\n📊 Evaluating at iteration {self.iter}...")
        
        evaluator = COCOEvaluator(f"{self.dataset_name}_val", self.cfg, False, output_dir=self.cfg.OUTPUT_DIR)
        val_loader = build_detection_test_loader(self.cfg, f"{self.dataset_name}_val")
        results = inference_on_dataset(self.model, val_loader, evaluator)
        
        current_ap = results['bbox']['AP50']
        print(f"   Current AP50: {current_ap:.2f}%")
        
        # Save best model
        if current_ap > self.best_val_ap:
            self.best_val_ap = current_ap
            self.patience_counter = 0
            
            # Save best model separately
            DetectionCheckpointer(self.model).save("best_model")
            print(f"   ✅ New best model! AP50: {current_ap:.2f}%")
        else:
            self.patience_counter += 1
            print(f"   No improvement for {self.patience_counter} evaluations")
        
        # Early stopping
        if self.patience_counter >= self.patience and self.iter > 10000:
            print(f"\n🛑 Early stopping triggered! No improvement for {self.patience} evaluations.")
            print(f"   Best AP50: {self.best_val_ap:.2f}%")
            self._trainer.stop()

# Check for existing checkpoints
checkpoint_files = glob.glob(os.path.join(cfg.OUTPUT_DIR, "model_*.pth"))
checkpoint_files.sort(key=os.path.getmtime, reverse=True)

resume_from = False
latest_iter = 0

if checkpoint_files:
    latest_checkpoint = checkpoint_files[0]
    print(f"✅ Found checkpoint: {os.path.basename(latest_checkpoint)}")
    match = re.search(r'model_(\d+)\.pth', latest_checkpoint)
    if match:
        latest_iter = int(match.group(1))
        if latest_iter > 100:
            resume_from = True
            print(f"🔄 Will RESUME from iteration {latest_iter}")
else:
    print("🆕 Starting fresh training")

# Initialize trainer
trainer = EnhancedTrainer(cfg)

if resume_from:
    DetectionCheckpointer(trainer.model).load(latest_checkpoint)
    print("✅ Checkpoint loaded")
else:
    trainer.resume_or_load(resume=False)

# Check GPU
print(f"\n🎮 GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Start training
start_time = time.time()
print(f"\n🔥 Training started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Target: {cfg.SOLVER.MAX_ITER} iterations (8 epochs)")
print(f"📊 Current: iteration {latest_iter}")
print(f"\n💡 ENHANCED FEATURES:")
print("   ✅ Auto-resume training")
print("   ✅ Early stopping (patience=5)")
print("   ✅ Best model saving")
print("   ✅ Learning rate warmup")

try:
    trainer.train()
    
    duration = (time.time() - start_time) / 3600
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print("="*70)
    print(f"⏱️ Training time: {duration:.2f} hours")
    print(f"💾 Final model: {cfg.OUTPUT_DIR}/model_final.pth")
    print(f"🏆 Best model: {cfg.OUTPUT_DIR}/best_model.pth (AP50: {trainer.best_val_ap:.2f}%)")
    
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted - checkpoints saved")
except Exception as e:
    print(f"\n❌ Error: {e}")

print("="*70)

TRAINING FASTER R-CNN MODEL (ENHANCED)
⚠️ This will take 5-6 hours!
🆕 Starting fresh training
[04/14 09:36:29 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=Fals

R-50.pkl: 102MB [00:00, 286MB/s]                            


[04/14 09:36:30 d2.checkpoint.c2_model_loading]: Renaming Caffe2 weights ......
[04/14 09:36:30 d2.checkpoint.c2_model_loading]: Following weights matched with submodule backbone.bottom_up - Total num: 54


Some model parameters or buffers are not found in the checkpoint:
backbone.fpn_lateral2.{bias, weight}
backbone.fpn_lateral3.{bias, weight}
backbone.fpn_lateral4.{bias, weight}
backbone.fpn_lateral5.{bias, weight}
backbone.fpn_output2.{bias, weight}
backbone.fpn_output3.{bias, weight}
backbone.fpn_output4.{bias, weight}
backbone.fpn_output5.{bias, weight}
proposal_generator.rpn_head.anchor_deltas.{bias, weight}
proposal_generator.rpn_head.conv.{bias, weight}
proposal_generator.rpn_head.objectness_logits.{bias, weight}
roi_heads.box_head.fc1.{bias, weight}
roi_heads.box_head.fc2.{bias, weight}
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
The checkpoint state_dict contains keys that are not used by the model:
  fc1000.{bias, weight}
  stem.conv1.bias



🎮 GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4

🔥 Training started at: 2026-04-14 09:36:30
📊 Target: 37500 iterations (8 epochs)
📊 Current: iteration 0

💡 ENHANCED FEATURES:
   ✅ Auto-resume training
   ✅ Early stopping (patience=5)
   ✅ Best model saving
   ✅ Learning rate warmup
[04/14 09:36:30 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0414 09:36:33.551000 55 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[04/14 09:36:47 d2.utils.events]:  eta: 7:17:27  iter: 19  total_loss: 1.942  loss_cls: 0.8928  loss_box_reg: 0.05908  loss_rpn_cls: 0.6967  loss_rpn_loc: 0.3238    time: 0.6988  last_time: 0.6955  data_time: 0.0228  last_data_time: 0.0089   lr: 4.8702e-06  max_mem: 3073M


2026-04-14 09:36:50.631492: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776159410.835677      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776159410.901211      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776159411.370118      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776159411.370158      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776159411.370164      55 computation_placer.cc:177] computation placer alr

[04/14 09:37:25 d2.utils.events]:  eta: 7:24:52  iter: 39  total_loss: 1.533  loss_cls: 0.4295  loss_box_reg: 0.05402  loss_rpn_cls: 0.6895  loss_rpn_loc: 0.309    time: 0.7066  last_time: 0.6791  data_time: 0.0172  last_data_time: 0.0064   lr: 9.8652e-06  max_mem: 3073M
[04/14 09:37:39 d2.utils.events]:  eta: 7:28:34  iter: 59  total_loss: 1.27  loss_cls: 0.1751  loss_box_reg: 0.06629  loss_rpn_cls: 0.6942  loss_rpn_loc: 0.3052    time: 0.7121  last_time: 0.7434  data_time: 0.0145  last_data_time: 0.0288   lr: 1.486e-05  max_mem: 3073M
[04/14 09:37:54 d2.utils.events]:  eta: 7:31:32  iter: 79  total_loss: 1.179  loss_cls: 0.1133  loss_box_reg: 0.06326  loss_rpn_cls: 0.6954  loss_rpn_loc: 0.2993    time: 0.7174  last_time: 0.7486  data_time: 0.0161  last_data_time: 0.0121   lr: 1.9855e-05  max_mem: 3073M
[04/14 09:38:09 d2.utils.events]:  eta: 7:34:22  iter: 99  total_loss: 1.152  loss_cls: 0.1021  loss_box_reg: 0.07967  loss_rpn_cls: 0.6879  loss_rpn_loc: 0.2813    time: 0.7249  last_

# **BLOCK 7: EVALUATE ON TEST SET**

In [10]:
# ============================================================================
# BLOCK 7: EVALUATE ON TEST SET (FIXED)
# ============================================================================

print("="*70)
print("EVALUATING MODEL ON TEST SET")
print("="*70)

from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# Load best model
best_model = os.path.join(cfg.OUTPUT_DIR, "best_model.pth")
if os.path.exists(best_model):
    cfg.MODEL.WEIGHTS = best_model
    print(f"✅ Using best model: {best_model}")
else:
    cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
    print(f"✅ Using final model: {cfg.MODEL.WEIGHTS}")

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
predictor = DefaultPredictor(cfg)

# Evaluate on test set
evaluator = COCOEvaluator(f"{dataset_name}_test", cfg, False, output_dir=cfg.OUTPUT_DIR)
test_loader = build_detection_test_loader(cfg, f"{dataset_name}_test")
results = inference_on_dataset(predictor.model, test_loader, evaluator)

print("\n" + "="*70)
print("📊 TEST RESULTS")
print("="*70)

# Store results safely (handles different metric names)
bbox_results = results['bbox']
test_results = {
    'AP': bbox_results.get('AP', 0),
    'AP50': bbox_results.get('AP50', 0),
    'AP75': bbox_results.get('AP75', 0),
    'AR1': bbox_results.get('AR@1', bbox_results.get('AR1', 0)),
    'AR10': bbox_results.get('AR@10', bbox_results.get('AR10', 0)),
    'AR100': bbox_results.get('AR@100', bbox_results.get('AR100', 0))
}

print(f"\n🎯 Average Precision (AP):")
print(f"   AP @ IoU=0.50:0.95: {test_results['AP']:.2f}%")
print(f"   AP50 @ IoU=0.50: {test_results['AP50']:.2f}%")
print(f"   AP75 @ IoU=0.75: {test_results['AP75']:.2f}%")

print(f"\n📈 Average Recall (AR):")
print(f"   AR @ 1 detection: {test_results['AR1']:.2f}%")
print(f"   AR @ 10 detections: {test_results['AR10']:.2f}%")
print(f"   AR @ 100 detections: {test_results['AR100']:.2f}%")

# Save results
with open(f"{cfg.OUTPUT_DIR}/test_results.json", 'w') as f:
    json.dump(results, f, indent=2)

with open(f"{cfg.OUTPUT_DIR}/metrics.json", 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\n✅ Results saved to {cfg.OUTPUT_DIR}/test_results.json")
print("="*70)

EVALUATING MODEL ON TEST SET
✅ Using final model: /kaggle/working/shoulder_arm_model/model_final.pth
[04/14 20:07:44 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/shoulder_arm_model/model_final.pth ...
WARNING [04/14 20:07:44 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
WARNING [04/14 20:07:44 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[04/14 20:07:44 d2.data.datasets.coco]: Loaded 3661 images in COCO format from /kaggle/working/test_coco.json
[04/14 20:07:44 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=800, sample_style='choice')]
[04/14 20:07:44 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[04/14 20:07:44 d2.data.common]: Seriali

# **BLOCK 8: GENERATE FORMAL PDF REPORT**

In [12]:
# ============================================================================
# BLOCK 8: GENERATE FORMAL PDF REPORT
# ============================================================================

print("="*70)
print("GENERATING FORMAL PDF REPORT")
print("="*70)

import shutil  # ← ADD THIS IMPORT
import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle

# Install reportlab for PDF generation
!pip install -q reportlab

# Create PDF document
pdf_path = f"{cfg.OUTPUT_DIR}/FINAL_RESULTS_REPORT.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=72,
    leftMargin=72,
    topMargin=72,
    bottomMargin=72
)

# Styles
styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'CustomTitle',
    parent=styles['Heading1'],
    fontSize=24,
    textColor=colors.HexColor('#1a365d'),
    spaceAfter=30,
    alignment=1
)
heading1_style = ParagraphStyle(
    'CustomHeading1',
    parent=styles['Heading1'],
    fontSize=18,
    textColor=colors.HexColor('#2b6cb0'),
    spaceAfter=12,
    spaceBefore=20
)
heading2_style = ParagraphStyle(
    'CustomHeading2',
    parent=styles['Heading2'],
    fontSize=14,
    textColor=colors.HexColor('#2d3748'),
    spaceAfter=8,
    spaceBefore=15
)
normal_style = ParagraphStyle(
    'CustomNormal',
    parent=styles['Normal'],
    fontSize=11,
    leading=16,
    spaceAfter=6
)

# Content list
story = []

# Title
story.append(Paragraph("AETHEA Bone Fracture Detection", title_style))
story.append(Paragraph("Faster R-CNN Model - Technical Report", title_style))
story.append(Spacer(1, 0.5*inch))

# Date
date_str = datetime.datetime.now().strftime("%B %d, %Y")
story.append(Paragraph(f"Report Date: {date_str}", normal_style))
story.append(Spacer(1, 0.2*inch))

# Executive Summary
story.append(Paragraph("1. Executive Summary", heading1_style))
story.append(Paragraph(
    "This report presents the results of training a Faster R-CNN model for "
    "shoulder and arm bone fracture detection on X-ray images. The model was "
    "trained on a filtered dataset containing only shoulder and arm fracture images.",
    normal_style
))

# Dataset Overview
story.append(Paragraph("2. Dataset Overview", heading1_style))

# Get dataset counts (use your actual numbers)
train_count = 18987
val_count = 4832
test_count = 3661

data_table_data = [
    ["Split", "Images"],
    ["Training", f"{train_count:,}"],
    ["Validation", f"{val_count:,}"],
    ["Test", f"{test_count:,}"],
    ["Total", f"{train_count + val_count + test_count:,}"]
]

data_table = Table(data_table_data, colWidths=[2*inch, 2*inch])
data_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2b6cb0')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
    ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor('#f7fafc')),
    ('GRID', (0, 0), (-1, -1), 1, colors.HexColor('#e2e8f0')),
]))
story.append(data_table)
story.append(Spacer(1, 0.2*inch))

# Performance Metrics
story.append(Paragraph("3. Performance Metrics", heading1_style))

# Use your actual test results
ap50 = 40.02  # Your actual AP50 from test results
ap = 15.10    # Your actual AP from test results
ap75 = 9.42   # Your actual AP75 from test results

metrics_table_data = [
    ["Metric", "Value"],
    ["AP50 (IoU=0.50)", f"{ap50:.2f}%"],
    ["AP (IoU=0.50:0.95)", f"{ap:.2f}%"],
    ["AP75 (IoU=0.75)", f"{ap75:.2f}%"],
]

metrics_table = Table(metrics_table_data, colWidths=[2.5*inch, 2*inch])
metrics_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2b6cb0')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
    ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor('#f7fafc')),
    ('GRID', (0, 0), (-1, -1), 1, colors.HexColor('#e2e8f0')),
]))
story.append(metrics_table)
story.append(Spacer(1, 0.2*inch))

# Conclusion
story.append(Paragraph("4. Conclusion", heading1_style))
story.append(Paragraph(
    f"The Faster R-CNN model achieved an AP50 of {ap50:.2f}% on the test set. "
    "The model is ready for deployment in the AETHEA medical platform.",
    normal_style
))

# Footer
story.append(Spacer(1, 0.5*inch))
story.append(Paragraph("-"*70, normal_style))
story.append(Spacer(1, 0.2*inch))
story.append(Paragraph(f"Report Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", normal_style))
story.append(Paragraph("Author: Andrew Wageh | Project: AETHEA Graduation Project", normal_style))

# Build PDF
doc.build(story)
print(f"✅ PDF report generated: {pdf_path}")

# Copy to output directory for download
shutil.copy(pdf_path, f"{WORKING_DIR}/FINAL_RESULTS_REPORT.pdf")

print(f"\n📁 PDF location: {pdf_path}")
print(f"📁 Also saved to: {WORKING_DIR}/FINAL_RESULTS_REPORT.pdf")
print("="*70)

GENERATING FORMAL PDF REPORT
✅ PDF report generated: /kaggle/working/shoulder_arm_model/FINAL_RESULTS_REPORT.pdf

📁 PDF location: /kaggle/working/shoulder_arm_model/FINAL_RESULTS_REPORT.pdf
📁 Also saved to: /kaggle/working/FINAL_RESULTS_REPORT.pdf
